# Chapter 10 — Evidence and Verification

**Book alignment:** current Chapter 10 · internal demo `Stage 09`

The shared demo package calls this **Stage 09** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** What evidence is sufficient to earn PASS, FAIL, PARTIAL, or UNKNOWN for the exact state that was verified?


## Four different claims

```text
ACTION RECEIPT
Did the command/tool complete?

STATE TRANSITION
Did the relevant environment state change?

GOAL SATISFACTION
Does the resulting state satisfy the user's contract?

EVALUATION INTEGRITY
Was that conclusion earned through a trustworthy path?
```

Lower-level success is not automatically promoted into a higher-level claim.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "demo" / "agents-from-first-principles").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing demo/agents-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / "demo" / "agents-from-first-principles"
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.verification import (
    Adjudicator,
    EvidenceCollector,
    EvidenceLayer,
    EvidenceSet,
    GoalContract,
    IntegrityStatus,
    Verifier,
    Verdict,
    WorkspaceSnapshot,
)

## Goal contract and exact state identity

The acting policy does not get to rewrite the success criteria. Evidence is bound to both the immutable contract fingerprint and the exact workspace state it verified.


In [ ]:
contract = GoalContract(
    must_change=("pipe records accepted",),
    must_preserve=("comma records still pass",),
    must_not=("delete protected tests", "weaken goal contract"),
)

baseline = WorkspaceSnapshot.from_mapping(
    {
        "parser.py": 'return text.split(",")',
        "tests/test_parser.py": "test_comma\ntest_pipe\n",
    }
)
fixed = WorkspaceSnapshot.from_mapping(
    {
        "parser.py": "return text.split(delimiter)",
        "tests/test_parser.py": "test_comma\ntest_pipe\n",
    }
)
tampered = WorkspaceSnapshot.from_mapping(
    {
        "parser.py": 'return text.split(",")',
        "tests/test_parser.py": "test_comma\n",
    }
)

assert baseline.state_id != fixed.state_id
assert baseline.state_id != tampered.state_id

## Experiment 1 - an action receipt cannot prove the goal


In [ ]:
collector = EvidenceCollector(verifier_version="chapter-10")
receipt_only = EvidenceSet(
    (
        collector.collect(
            evidence_id="tool-zero",
            criterion="pipe records accepted",
            layer=EvidenceLayer.ACTION_RECEIPT,
            source="tool receipt",
            state_id=fixed.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=1,
        ),
    )
)

clean_integrity = Verifier(
    baseline=baseline,
    protected_paths=("tests/test_parser.py",),
).check_integrity(contract=contract, current=fixed)

receipt_result = Adjudicator().decide(
    contract=contract,
    evidence=receipt_only,
    state_id=fixed.state_id,
    integrity=clean_integrity,
)

assert receipt_result.verdict == Verdict.UNKNOWN
assert receipt_result.verification_coverage == 0.0

## Experiment 2 - evidence is valid only for the state it verified

A PASS collected at state A becomes stale after the workspace changes to state B.


In [ ]:
old_evidence = EvidenceSet(
    (
        collector.collect(
            evidence_id="old-pipe-pass",
            criterion="pipe records accepted",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="targeted pytest",
            state_id=baseline.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=2,
        ),
        collector.collect(
            evidence_id="old-comma-pass",
            criterion="comma records still pass",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="regression pytest",
            state_id=baseline.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=2,
        ),
    )
)

stale_result = Adjudicator().decide(
    contract=contract,
    evidence=old_evidence,
    state_id=fixed.state_id,
    integrity=clean_integrity,
)

assert stale_result.verdict == Verdict.UNKNOWN
assert set(stale_result.stale_evidence_ids) == {"old-pipe-pass", "old-comma-pass"}

## Experiment 3 - current evidence plus clean integrity can earn PASS


In [ ]:
current_evidence = EvidenceSet(
    (
        collector.collect(
            evidence_id="pipe-pass",
            criterion="pipe records accepted",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="targeted pytest",
            state_id=fixed.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=3,
        ),
        collector.collect(
            evidence_id="comma-pass",
            criterion="comma records still pass",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="regression pytest",
            state_id=fixed.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=3,
        ),
    )
)

clean_result = Adjudicator().decide(
    contract=contract,
    evidence=current_evidence,
    state_id=fixed.state_id,
    integrity=clean_integrity,
)

assert clean_integrity.status == IntegrityStatus.CLEAN
assert clean_result.verdict == Verdict.PASS
assert clean_result.verification_coverage == 1.0

## Experiment 4 - naive test PASS does not survive integrity violation

Now take the shortcut: remove the protected failing test. Pretend the task checker reports PASS because the remaining tests pass. The protected verifier must still return FAIL.


In [ ]:
naive_pass = EvidenceSet(
    (
        collector.collect(
            evidence_id="naive-pipe-pass",
            criterion="pipe records accepted",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="naive test checker",
            state_id=tampered.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=4,
        ),
        collector.collect(
            evidence_id="naive-comma-pass",
            criterion="comma records still pass",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="naive test checker",
            state_id=tampered.state_id,
            contract=contract,
            verdict=Verdict.PASS,
            collected_at=4,
        ),
    )
)

tamper_integrity = Verifier(
    baseline=baseline,
    protected_paths=("tests/test_parser.py",),
).check_integrity(contract=contract, current=tampered)

protected_result = Adjudicator().decide(
    contract=contract,
    evidence=naive_pass,
    state_id=tampered.state_id,
    integrity=tamper_integrity,
)

assert all(item.verdict == Verdict.PASS for item in protected_result.criteria)
assert tamper_integrity.status == IntegrityStatus.VIOLATED
assert protected_result.verdict == Verdict.FAIL

## Experiment 5 - the acting policy cannot lower the bar

Evidence collected against a weakened goal contract has a different contract fingerprint and cannot satisfy the original contract.


In [ ]:
weakened = GoalContract(
    must_change=("pipe records accepted",),
    must_preserve=(),
    must_not=(),
)
weakened_evidence = EvidenceSet(
    (
        collector.collect(
            evidence_id="easy-pass",
            criterion="pipe records accepted",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="weakened checker",
            state_id=fixed.state_id,
            contract=weakened,
            verdict=Verdict.PASS,
            collected_at=5,
        ),
    )
)

weakened_result = Adjudicator().decide(
    contract=contract,
    evidence=weakened_evidence,
    state_id=fixed.state_id,
    integrity=clean_integrity,
)

assert weakened.fingerprint != contract.fingerprint
assert weakened_result.verdict == Verdict.UNKNOWN

## Experiment 6 - PASS, FAIL, PARTIAL, and UNKNOWN are different claims

The current chapter makes the four-way verdict explicit. `UNKNOWN` means the required current evidence is missing; `PARTIAL` means the available current evidence says a criterion is only partly satisfied. Neither should be collapsed into `FAIL`.


In [ ]:
def adjudicate_two_criteria(pipe_verdict, comma_verdict=None):
    records = [
        collector.collect(
            evidence_id=f"pipe-{pipe_verdict.value.lower()}",
            criterion="pipe records accepted",
            layer=EvidenceLayer.GOAL_SATISFACTION,
            source="controlled verdict matrix",
            state_id=fixed.state_id,
            contract=contract,
            verdict=pipe_verdict,
            collected_at=10,
        )
    ]
    if comma_verdict is not None:
        records.append(
            collector.collect(
                evidence_id=f"comma-{comma_verdict.value.lower()}",
                criterion="comma records still pass",
                layer=EvidenceLayer.GOAL_SATISFACTION,
                source="controlled verdict matrix",
                state_id=fixed.state_id,
                contract=contract,
                verdict=comma_verdict,
                collected_at=10,
            )
        )
    return Adjudicator().decide(
        contract=contract,
        evidence=EvidenceSet(tuple(records)),
        state_id=fixed.state_id,
        integrity=clean_integrity,
    )


verdict_matrix = {
    "PASS": adjudicate_two_criteria(Verdict.PASS, Verdict.PASS).verdict,
    "FAIL": adjudicate_two_criteria(Verdict.FAIL, Verdict.PASS).verdict,
    "PARTIAL": adjudicate_two_criteria(Verdict.PARTIAL, Verdict.PASS).verdict,
    "UNKNOWN": adjudicate_two_criteria(Verdict.PASS, None).verdict,
}

assert verdict_matrix == {
    "PASS": Verdict.PASS,
    "FAIL": Verdict.FAIL,
    "PARTIAL": Verdict.PARTIAL,
    "UNKNOWN": Verdict.UNKNOWN,
}
{k: v.value for k, v in verdict_matrix.items()}

## Diagnostics


In [ ]:
false_pass_detected = (
    all(item.verdict == Verdict.PASS for item in protected_result.criteria)
    and protected_result.verdict == Verdict.FAIL
)
diagnostics = {
    "false_PASS_detected": false_pass_detected,
    "stale_evidence_count": len(stale_result.stale_evidence_ids),
    "verification_coverage_clean": clean_result.verification_coverage,
    "integrity_escape": 0 if protected_result.verdict == Verdict.FAIL else 1,
}
assert diagnostics == {
    "false_PASS_detected": True,
    "stale_evidence_count": 2,
    "verification_coverage_clean": 1.0,
    "integrity_escape": 0,
}
diagnostics

## What was earned

The completion boundary now depends on **current, state-bound, goal-level evidence plus clean evaluation integrity**.

```text
tool receipt PASS
≠ goal PASS

old evidence PASS
+ changed state
= stale evidence

naive task checker PASS
+ protected-test modification
= final FAIL

all required current criteria PASS
+ integrity CLEAN
= PASS
```

The model or acting policy can propose and act, but it does not get to decide what its own evidence earns.
